In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

# Cargar las variables del archivo .env (API Keys y configuración)
load_dotenv()

LLM_API_KEY = os.getenv("LLM_API_KEY")
LLM_BASE_URL = "https://api.mistral.ai/v1"
LLM_MODEL = "ministral-8b-latest"
EMBEDDING_MODEL = "mistral-embed"

# Configuración para habilitar la trazabilidad en LangSmith (IL1.4)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "SP-Digital-RAG-Evaluation")

assert LLM_API_KEY, "Error: No se encontró la variable LLM_API_KEY en el archivo .env"

# Creamos el cliente de OpenAI apuntando a Mistral y envuelto para LangSmith
import time, random, threading
import httpx

class TransporteMistral(httpx.BaseTransport):
    """Capa HTTP compartida por TODAS las llamadas a Mistral (cliente OpenAI y ChatOpenAI de LangChain).
    1) Espacia las peticiones (intervalo minimo) para no provocar el 429.
    2) Si igual llega un 429/5xx, espera (respetando Retry-After, hasta 60 s) y reintenta SOLO esa
       peticion, sin reiniciar el agente ni repetir herramientas con efectos (cotizaciones, tickets)."""

    ESTADOS_REINTENTABLES = (429, 502, 503, 504)

    def __init__(self, intervalo_min=1.5, max_reintentos=6, espera_max=60, base=None):
        self.base = base or httpx.HTTPTransport()
        self.intervalo_min = intervalo_min
        self.max_reintentos = max_reintentos
        self.espera_max = espera_max
        self._lock = threading.Lock()
        self._ultima = 0.0

    def _esperar_turno(self):
        with self._lock:
            falta = self.intervalo_min - (time.monotonic() - self._ultima)
            if falta > 0:
                time.sleep(falta)
            self._ultima = time.monotonic()

    def _espera(self, resp, intento):
        ra = resp.headers.get("retry-after") if resp is not None else None
        try:
            if ra:
                return min(float(ra), self.espera_max) + random.uniform(0, 1)
        except ValueError:
            pass
        return min(self.espera_max, 3 * 2 ** intento) + random.uniform(0, 2)

    def handle_request(self, request):
        for intento in range(self.max_reintentos + 1):
            self._esperar_turno()
            try:
                resp = self.base.handle_request(request)
            except (httpx.ConnectError, httpx.ConnectTimeout):
                if intento == self.max_reintentos:
                    raise
                time.sleep(self._espera(None, intento))
                continue
            if resp.status_code not in self.ESTADOS_REINTENTABLES or intento == self.max_reintentos:
                return resp
            espera = self._espera(resp, intento)
            resp.close()
            print(f"  (HTTP {resp.status_code} de Mistral, esperando {espera:.0f}s y reintentando {intento + 1}/{self.max_reintentos}...)")
            time.sleep(espera)

    def close(self):
        self.base.close()

# Un unico cliente HTTP para todo el notebook (el SDK no reintenta: lo hace el transporte)
HTTP_MISTRAL = httpx.Client(transport=TransporteMistral(), timeout=httpx.Timeout(120.0))

client = wrap_openai(OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL, max_retries=0, http_client=HTTP_MISTRAL))
print("Cliente conectado a Mistral y trazabilidad de LangSmith activada correctamente.")

Cliente conectado a Mistral y trazabilidad de LangSmith activada correctamente.


In [2]:
import time, random
from openai import RateLimitError

def llamar_con_espera(fn, *args, max_reintentos=3, **kwargs):
    """Red de seguridad por encima del transporte HTTP.
    El transporte ya reintenta cada peticion con 429; aqui solo se reintenta si aun asi sale
    RateLimitError, con esperas crecientes (20s, 40s...) para dejar que se limpie la ventana de Mistral."""
    for intento in range(max_reintentos):
        try:
            return fn(*args, **kwargs)
        except RateLimitError:
            if intento == max_reintentos - 1:
                break
            espera = min(60, 20 * (intento + 1)) + random.uniform(0, 3)
            print(f"  (429 persistente de Mistral, esperando {espera:.0f}s y reintentando {intento + 1}/{max_reintentos - 1}...)")
            time.sleep(espera)
    raise RuntimeError(
        "Mistral siguio respondiendo 429. Revisa tu cuota/limites en https://console.mistral.ai "
        "(si la cuota mensual o diaria esta agotada, esperar no ayuda) o prueba otro modelo con LLM_MODEL_AGENTE."
    )


In [3]:
# Catálogo de productos de prueba para SP Digital
CATALOGO = [
    {
        "sku": "GPU-4070-ASUS",
        "nombre": "ASUS Dual RTX 4070 OC 12GB",
        "categoria": "GPU",
        "marca": "ASUS",
        "specs": "GPU NVIDIA GeForce RTX 4070, 12GB GDDR6X, conector 1x 16-pin (12VHPWR), consumo tipico 200W, fuente recomendada: 650W minimo, PCIe 4.0 x16."
    },
    {
        "sku": "MB-B450-MSI",
        "nombre": "MSI B450 Tomahawk Max",
        "categoria": "Placa madre",
        "marca": "MSI",
        "specs": "Placa madre socket AM4, chipset B450, soporta Ryzen serie 1000 a 5000, 1x slot PCIe 3.0 x16, soporta memoria DDR4."
    },
    {
        "sku": "MB-B650-ASUS",
        "nombre": "ASUS TUF Gaming B650-PLUS WiFi",
        "categoria": "Placa madre",
        "marca": "ASUS",
        "specs": "Placa madre socket AM5, chipset B650, soporta Ryzen serie 7000/8000/9000, 1x slot PCIe 4.0 x16, soporta solo memoria DDR5."
    },
    {
        "sku": "CPU-R5-5600X",
        "nombre": "AMD Ryzen 5 5600X",
        "categoria": "Procesador",
        "marca": "AMD",
        "specs": "Procesador Socket AM4, 6 nucleos, 12 hilos, TDP 65W, no incluye graficos integrados, compatible con memoria DDR4."
    },
    {
        "sku": "CPU-R7-7800X3D",
        "nombre": "AMD Ryzen 7 7800X3D",
        "categoria": "Procesador",
        "marca": "AMD",
        "specs": "Procesador Socket AM5, 8 nucleos, 16 hilos, TDP 120W, requiere cooler, compatible con memoria DDR5."
    },
    {
        "sku": "RAM-DDR4-16GB-CORSAIR",
        "nombre": "Corsair Vengeance LPX 16GB (2x8GB) DDR4 3200MHz",
        "categoria": "Memoria RAM",
        "marca": "Corsair",
        "specs": "Memoria RAM DDR4, capacidad 16GB (2x8GB), frecuencia 3200MHz, formato UDIMM para PC de escritorio."
    },
    {
        "sku": "RAM-DDR5-32GB-KINGSTON",
        "nombre": "Kingston Fury Beast 32GB (2x16GB) DDR5 5600MHz",
        "categoria": "Memoria RAM",
        "marca": "Kingston",
        "specs": "Memoria RAM DDR5, capacidad 32GB (2x16GB), frecuencia 5600MHz, formato UDIMM para PC de escritorio."
    },
    {
        "sku": "PSU-CORSAIR-650",
        "nombre": "Corsair RM650x 650W 80+ Gold",
        "categoria": "Fuente de poder",
        "marca": "Corsair",
        "specs": "Fuente 650W, 80+ Gold, modular, 1x conector 12VHPWR (con adaptador), 2x conectores PCIe 8-pin."
    },
    {
        "sku": "PSU-EVGA-450",
        "nombre": "EVGA 450 BR 450W 80+ Bronze",
        "categoria": "Fuente de poder",
        "marca": "EVGA",
        "specs": "Fuente 450W, 80+ Bronze, no modular, 2x conectores PCIe 8-pin (6+2). No posee conector 12VHPWR."
    }
]

# Reglas duras de compatibilidad para validar las respuestas del LLM
REGLAS_COMPATIBILIDAD = {
    ("GPU-4070-ASUS", "MB-B450-MSI"): {
        "compatible": True,
        "motivo": "El slot PCIe 3.0 x16 acepta la GPU (PCIe 4.0 es retrocompatible).",
        "requisito_adicional": "Fuente >= 650W con conector 12VHPWR o adaptador."
    },
    ("CPU-R7-7800X3D", "MB-B450-MSI"): {
        "compatible": False,
        "motivo": "Incompatibilidad de socket: El procesador es AM5 y la placa madre es AM4.",
        "requisito_adicional": "Requiere una placa madre con Socket AM5 (ej. B650/X670)."
    },
    ("RAM-DDR5-32GB-KINGSTON", "MB-B450-MSI"): {
        "compatible": False,
        "motivo": "Incompatibilidad de memoria: La placa MSI B450 solo soporta DDR4, no DDR5.",
        "requisito_adicional": "Usar memoria DDR4 o cambiar a placa AM5/Intel compatible con DDR5."
    },
    ("GPU-4070-ASUS", "PSU-EVGA-450"): {
        "compatible": False,
        "motivo": "Potencia insuficiente: La GPU requiere minimo 650W y la fuente es de 450W.",
        "requisito_adicional": "Cambiar fuente a minimo 650W 80+ Gold con soporte 12VHPWR."
    }
}

# Documentos técnicos adicionales para alimentar el contexto del RAG
NOTAS_TECNICAS = [
    {
        "sku": "NOTA-PCIE-RETROCOMPAT",
        "nombre": "Nota tecnica: retrocompatibilidad PCIe",
        "categoria": "Nota tecnica",
        "marca": "N/A",
        "specs": "Los slots y tarjetas PCI Express (PCIe) son retrocompatibles entre "
                 "generaciones: una tarjeta PCIe 4.0 funciona correctamente en un slot "
                 "PCIe 3.0, y viceversa. La diferencia de generacion no impide el "
                 "funcionamiento, solo limita el ancho de banda maximo teorico; en la "
                 "practica esto casi no afecta el rendimiento de una GPU en uso gamer."
    }
]

print(f"Cargados: {len(CATALOGO)} productos y {len(NOTAS_TECNICAS)} notas técnicas.")

Cargados: 9 productos y 1 notas técnicas.


In [4]:
# Función para convertir los datos en bloques de texto (chunks) para el RAG
def construir_chunks(catalogo):
    chunks = []
    for p in catalogo:
        texto = f"Producto: {p['nombre']} (SKU: {p['sku']}). Categoria: {p['categoria']}. " \
                f"Marca: {p['marca']}. Especificaciones: {p['specs']}"
        chunks.append({"texto": texto, "metadata": {"sku": p["sku"]}})
    return chunks

CHUNKS = construir_chunks(CATALOGO + NOTAS_TECNICAS)
print("SKUs procesados para vectorización:")
for c in CHUNKS:
    print("-", c["metadata"]["sku"])

SKUs procesados para vectorización:
- GPU-4070-ASUS
- MB-B450-MSI
- MB-B650-ASUS
- CPU-R5-5600X
- CPU-R7-7800X3D
- RAM-DDR4-16GB-CORSAIR
- RAM-DDR5-32GB-KINGSTON
- PSU-CORSAIR-650
- PSU-EVGA-450
- NOTA-PCIE-RETROCOMPAT


In [5]:
import numpy as np
import faiss

# Función para sacar los vectores embeddings de los textos con Mistral
def obtener_embeddings(textos):
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=textos)
    return np.array([d.embedding for d in resp.data], dtype="float32")

# Creamos el índice de búsqueda vectorial FAISS
embeddings_catalogo = obtener_embeddings([c["texto"] for c in CHUNKS])
faiss.normalize_L2(embeddings_catalogo)
index = faiss.IndexFlatIP(embeddings_catalogo.shape[1])
index.add(embeddings_catalogo)

print("Índice FAISS listo con", index.ntotal, "vectores cargados.")

Índice FAISS listo con 10 vectores cargados.


In [6]:
# Módulo de recuperación RAG (retriever) monitoreado por LangSmith
@traceable(name="FAISS_Vector_Retrieval", run_type="retriever")
def recuperar_contexto(pregunta, top_k=3):
    q_emb = obtener_embeddings([pregunta])
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, top_k)
    return [{**CHUNKS[i], "score": float(s)} for s, i in zip(scores[0], idxs[0]) if i != -1]

In [7]:
# Prompt del sistema con instrucciones estrictas para evitar alucinaciones
SYSTEM_PROMPT = """Eres el asistente tecnico de compatibilidad de SP Digital.
1. Responde SOLO usando la informacion en <contexto>.
2. Si falta el dato, indica que no puedes confirmar y deriva a un vendedor humano.
3. Nunca inventes sockets, wattajes, chipsets ni modelos que no aparezcan
   textualmente en <contexto>. Esto incluye la seccion de Recomendacion:
   si vas a sugerir un producto o especificacion alternativa, esta debe
   estar en <contexto>; si no lo esta, di explicitamente que no tienes
   alternativas disponibles en el catalogo consultado y deriva a un
   vendedor humano para evaluar opciones fuera del catalogo.
4. Responde en 3 partes: Veredicto / Justificacion / Recomendacion.
5. Tono tecnico y breve (max. 120 palabras)."""

def construir_prompt(pregunta, fragmentos):
    contexto = "\n\n".join(f"[{f['metadata']['sku']}] {f['texto']}" for f in fragmentos)
    return f'<contexto>\n{contexto}\n</contexto>\n\nPregunta: "{pregunta}"'

In [8]:
# Función auxiliar para hacer consultas de prueba rápidas por consola
def preguntar(pregunta):
    fragmentos = recuperar_contexto(pregunta)
    prompt = construir_prompt(pregunta, fragmentos)
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2, max_tokens=250,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": prompt}],
    )
    print("CONTEXTO USADO:", [f["metadata"]["sku"] for f in fragmentos])
    print("\nRESPUESTA:\n", resp.choices[0].message.content)

In [9]:
preguntar("Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?")

CONTEXTO USADO: ['MB-B450-MSI', 'GPU-4070-ASUS', 'NOTA-PCIE-RETROCOMPAT']

RESPUESTA:
 **Veredicto:** **Sí, compatible.**

**Justificación:** La MSI B450 Tomahawk Max usa **socket AM4** y soporta **PCIe 3.0 x16**, mientras que la ASUS RTX 4070 es **PCIe 4.0 x16**. Según la nota técnica, las tarjetas PCIe 4.0 funcionan en slots PCIe 3.0 sin problemas prácticos en rendimiento gamer (solo limita ancho de banda teórico). Además, el conector **12VHPWR** es compatible con fuentes de 650W+ (recomendado por ASUS).

**Recomendación:** Verifica que tu fuente tenga **650W mínimo** y conector **12VHPWR** para evitar incompatibilidades de alimentación. No se requieren ajustes adicionales.


In [10]:
preguntar("Que fuente de poder necesito para la RTX 4070 si mi fuente actual es de 450W?")

CONTEXTO USADO: ['GPU-4070-ASUS', 'PSU-EVGA-450', 'PSU-CORSAIR-650']

RESPUESTA:
 **Veredicto:** **Incompatible.** La GPU requiere **12VHPWR** y **650W mínimo**, pero tu fuente actual (EVGA 450W) no cumple con ninguno.

**Justificación:**
- La **RTX 4070 ASUS** exige **12VHPWR** (no compatible con conectores PCIe 8-pin estándar) y **650W mínimo** para operar sin riesgos.
- La **EVGA 450W** solo ofrece **2x PCIe 8-pin** y **450W**, insuficiente en potencia y tipo de conector.

**Recomendación:**
Usa la **Corsair RM650x (650W, 80+ Gold)**. Incluye **12VHPWR** (con adaptador) y supera el requerimiento de potencia. *No hay alternativas en el catálogo con 12VHPWR y ≥650W.* Deriva a vendedor humano para evaluar opciones fuera del catálogo.


In [11]:
# Módulo para cruzar los SKUs recuperados contra las reglas del negocio
def validar_contra_reglas(skus_en_contexto):
    for (a, b), regla in REGLAS_COMPATIBILIDAD.items():
        if a in skus_en_contexto and b in skus_en_contexto:
            return regla
    return None

# Probamos con un par que sí esté registrado
resultado = validar_contra_reglas(["GPU-4070-ASUS", "MB-B450-MSI"])
print("Regla encontrada:", resultado)

Regla encontrada: {'compatible': True, 'motivo': 'El slot PCIe 3.0 x16 acepta la GPU (PCIe 4.0 es retrocompatible).', 'requisito_adicional': 'Fuente >= 650W con conector 12VHPWR o adaptador.'}


In [12]:
# Pipeline principal RAG registrado en LangSmith para ver toda la trazabilidad
@traceable(name="Pipeline_RAG_SP_Digital", run_type="chain")
def ejecutar_pipeline_rag(pregunta):
    fragmentos = llamar_con_espera(recuperar_contexto, pregunta, top_k=3)
    time.sleep(2)
    prompt = construir_prompt(pregunta, fragmentos)

    resp = llamar_con_espera(
        client.chat.completions.create,
        model=LLM_MODEL,
        temperature=0.2,
        max_tokens=250,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
    )

    respuesta = resp.choices[0].message.content
    skus = [f["metadata"]["sku"] for f in fragmentos]
    regla = validar_contra_reglas(skus)

    return {
        "pregunta": pregunta,
        "respuesta_llm": respuesta,
        "skus_recuperados": skus,
        "regla_aplicada": regla,
        "fragmentos": fragmentos
    }

In [13]:
# Probamos el pipeline completo con una pregunta de prueba
resultado = ejecutar_pipeline_rag("Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?")

print("Pregunta:", resultado["pregunta"])
print("-" * 60)
print("Contexto (SKUs recuperados):", resultado["skus_recuperados"])
print("-" * 60)
print("Respuesta del modelo:\n", resultado["respuesta_llm"])
print("-" * 60)
print("Validación de regla:", resultado["regla_aplicada"])

Pregunta: Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?
------------------------------------------------------------
Contexto (SKUs recuperados): ['MB-B450-MSI', 'GPU-4070-ASUS', 'NOTA-PCIE-RETROCOMPAT']
------------------------------------------------------------
Respuesta del modelo:
 **Veredicto:** **Sí, compatible.**

**Justificación:** La placa **MSI B450 Tomahawk Max (AM4)** soporta **PCIe 3.0 x16**, y la **RTX 4070 (PCIe 4.0 x16)** es retrocompatible. La nota técnica confirma que la GPU funcionará sin problemas, aunque con ancho de banda limitado a **PCIe 3.0** (sin impacto práctico en rendimiento gamer).

**Recomendación:** Verifica que tu fuente de poder sea **≥650W** (recomendado por ASUS). No hay alternativas en el catálogo para mejorar compatibilidad.
------------------------------------------------------------
Validación de regla: {'compatible': True, 'motivo': 'El slot PCIe 3.0 x16 acepta la GPU (PCIe 4.0 es retrocompatible).', 'requisito_adicional': 

In [14]:
import re
import time

# Casos de prueba para evaluar la precisión del sistema frente al LLM directo
SET_PRUEBA = [
    {
        "pregunta": "Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?",
        "veredicto_esperado": "COMPATIBLE"
    },
    {
        "pregunta": "Puedo instalar un Ryzen 7 7800X3D en mi placa MSI B450 Tomahawk Max?",
        "veredicto_esperado": "NO COMPATIBLE"
    },
    {
        "pregunta": "Sirve la memoria RAM Kingston DDR5 de 32GB para la placa MSI B450?",
        "veredicto_esperado": "NO COMPATIBLE"
    },
    {
        "pregunta": "Puedo levantar la RTX 4070 con la fuente EVGA de 450W?",
        "veredicto_esperado": "NO COMPATIBLE"
    },
    {
        "pregunta": "Tienen disponibilidad de un cooler liquido de 360mm para gabinete ITX?",
        "veredicto_esperado": "SIN REGLA -> DERIVAR"
    }
]

def extraer_veredicto_llm(texto_respuesta):
    m = re.search(r"Veredicto:\**\s*✅?\s*\**\s*([^\n*]+)", texto_respuesta, re.IGNORECASE)
    texto = m.group(1).strip() if m else texto_respuesta[:40]
    if "no" in texto.lower() and "compat" in texto.lower():
        return "NO COMPATIBLE"
    if "compat" in texto.lower():
        return "COMPATIBLE"
    return "NO DETECTADO"

# llamar_con_espera se define una sola vez (celda 2); aqui NO se redefine.

def evaluar_caso(pregunta):
    fragmentos = llamar_con_espera(recuperar_contexto, pregunta, top_k=3)
    time.sleep(2)  # pausa entre el embedding y la respuesta del LLM
    prompt = construir_prompt(pregunta, fragmentos)
    resp = llamar_con_espera(
        client.chat.completions.create,
        model=LLM_MODEL, temperature=0.2, max_tokens=250,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": prompt}],
    )
    respuesta_llm = resp.choices[0].message.content
    skus = [f["metadata"]["sku"] for f in fragmentos]
    regla = validar_contra_reglas(skus)

    veredicto_llm = extraer_veredicto_llm(respuesta_llm)
    veredicto_sistema = ("COMPATIBLE" if regla["compatible"] else "NO COMPATIBLE") if regla else "SIN REGLA -> DERIVAR"

    return veredicto_llm, veredicto_sistema

aciertos_sistema = 0
coincidencias_llm = 0
for caso in SET_PRUEBA:
    veredicto_llm, veredicto_sistema = evaluar_caso(caso["pregunta"])
    time.sleep(3)  # pausa para respetar el límite de Mistral (cada caso hace 2 llamadas)
    ok_sistema = veredicto_sistema == caso["veredicto_esperado"]
    ok_llm_vs_sistema = veredicto_llm == veredicto_sistema
    aciertos_sistema += int(ok_sistema)
    coincidencias_llm += int(ok_llm_vs_sistema)

    print(f"Pregunta: {caso['pregunta']}")
    print(f"  Veredicto del LLM sin filtro : {veredicto_llm}")
    print(f"  Veredicto final del sistema  : {veredicto_sistema}  {'[OK]' if ok_sistema else '[FALLA]'} (Esperado: {caso['veredicto_esperado']})")
    if not ok_llm_vs_sistema:
        print(f"  -> El validador por regla corrigió al LLM (El LLM decía '{veredicto_llm}')")
    print()

print(f"Precisión del sistema final con validador: {aciertos_sistema}/{len(SET_PRUEBA)} ({100*aciertos_sistema/len(SET_PRUEBA):.0f}%)")
print(f"Coincidencia LLM directo vs sistema final: {coincidencias_llm}/{len(SET_PRUEBA)} ({100*coincidencias_llm/len(SET_PRUEBA):.0f}%)")

Pregunta: Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?
  Veredicto del LLM sin filtro : COMPATIBLE
  Veredicto final del sistema  : COMPATIBLE  [OK] (Esperado: COMPATIBLE)

  (429 de Mistral, esperando 5s y reintentando...)
Pregunta: Puedo instalar un Ryzen 7 7800X3D en mi placa MSI B450 Tomahawk Max?
  Veredicto del LLM sin filtro : NO COMPATIBLE
  Veredicto final del sistema  : NO COMPATIBLE  [OK] (Esperado: NO COMPATIBLE)

Pregunta: Sirve la memoria RAM Kingston DDR5 de 32GB para la placa MSI B450?
  Veredicto del LLM sin filtro : NO COMPATIBLE
  Veredicto final del sistema  : NO COMPATIBLE  [OK] (Esperado: NO COMPATIBLE)

  (429 de Mistral, esperando 5s y reintentando...)
Pregunta: Puedo levantar la RTX 4070 con la fuente EVGA de 450W?
  Veredicto del LLM sin filtro : NO DETECTADO
  Veredicto final del sistema  : NO COMPATIBLE  [OK] (Esperado: NO COMPATIBLE)
  -> El validador por regla corrigió al LLM (El LLM decía 'NO DETECTADO')

Pregunta: Tienen disponibili

In [15]:
import time
from langsmith import Client
from langsmith.evaluation import evaluate

# Dataset de prueba con las respuestas esperadas (Ground Truth) para la evaluación de RAG
DATASET_EVALUACION = [
    {
        "inputs": {"question": "Es compatible la ASUS RTX 4070 con mi placa MSI B450 Tomahawk Max?"},
        "outputs": {
            "expected_verdict": "COMPATIBLE",
            "expected_skus": ["MB-B450-MSI", "GPU-4070-ASUS"]
        }
    },
    {
        "inputs": {"question": "Puedo instalar un Ryzen 7 7800X3D en mi placa MSI B450 Tomahawk Max?"},
        "outputs": {
            "expected_verdict": "NO COMPATIBLE",
            "expected_skus": ["MB-B450-MSI", "CPU-R7-7800X3D"]
        }
    },
    {
        "inputs": {"question": "Sirve la memoria RAM Kingston DDR5 de 32GB para la placa MSI B450?"},
        "outputs": {
            "expected_verdict": "NO COMPATIBLE",
            "expected_skus": ["MB-B450-MSI", "RAM-DDR5-32GB-KINGSTON"]
        }
    },
    {
        "inputs": {"question": "Puedo levantar la RTX 4070 con la fuente EVGA de 450W?"},
        "outputs": {
            "expected_verdict": "NO COMPATIBLE",
            "expected_skus": ["GPU-4070-ASUS", "PSU-EVGA-450"]
        }
    },
    {
        "inputs": {"question": "Tienen disponibilidad de un cooler liquido de 360mm para gabinete ITX?"},
        "outputs": {
            "expected_verdict": "SIN REGLA -> DERIVAR",
            "expected_skus": []
        }
    }
]

# Evaluador 1: Context Recall (revisa si FAISS recuperó los SKUs necesarios)
def eval_context_recall(run, example):
    skus_obtenidos = run.outputs.get("skus_recuperados", [])
    skus_esperados = example.outputs.get("expected_skus", [])
    
    if not skus_esperados:
        score = 1.0
    else:
        coincidencias = set(skus_esperados).intersection(set(skus_obtenidos))
        score = len(coincidencias) / len(skus_esperados)
        
    return {"key": "rag_context_recall", "score": score}

# Evaluador 2: Faithfulness (usamos el modelo como juez para ver si la respuesta es fiel al contexto)
def eval_faithfulness_llm_judge(run, example):
    respuesta = run.outputs.get("respuesta_llm", "")
    fragmentos = run.outputs.get("fragmentos", [])
    contexto_str = "\n".join([f["texto"] for f in fragmentos])
    
    judge_prompt = f"""
    Eres un auditor técnico estricto. Evalúa si la siguiente 'Respuesta' está respaldada 100% por la información en 'Contexto'.
    Si la respuesta menciona datos, sockets, watts o compatibilidades que NO aparecen en 'Contexto', responde 'INFIEL'.
    Si la respuesta deriva a un humano porque falta información, responde 'FIEL'.
    Si todo está respaldado en el contexto, responde 'FIEL'.

    Contexto: {contexto_str}
    Respuesta: {respuesta}

    Resultado (responde SOLO la palabra FIEL o INFIEL):
    """

    time.sleep(2)
    judge_resp = llamar_con_espera(
        client.chat.completions.create,
        model=LLM_MODEL,
        temperature=0.0,
        messages=[{"role": "user", "content": judge_prompt}]
    )
    
    resultado = judge_resp.choices[0].message.content.strip().upper()
    score = 1.0 if "FIEL" in resultado else 0.0
    return {"key": "rag_faithfulness", "score": score}

# Evaluador 3: Accuracy del veredicto final
def eval_verdict_accuracy(run, example):
    regla = run.outputs.get("regla_aplicada")
    if regla:
        veredicto_sistema = "COMPATIBLE" if regla["compatible"] else "NO COMPATIBLE"
    else:
        veredicto_sistema = "SIN REGLA -> DERIVAR"
        
    esperado = example.outputs.get("expected_verdict")
    return {"key": "verdict_accuracy", "score": 1.0 if veredicto_sistema == esperado else 0.0}

def predict_rag(inputs):
    time.sleep(3)
    res = ejecutar_pipeline_rag(inputs["question"])
    return res

print("=== INICIANDO EVALUACIÓN RAG Y LANGSMITH ===")

try:
    ls_client = Client()
    dataset_name = "SP-Digital-RAG-Benchmark"
    
    if not ls_client.has_dataset(dataset_name=dataset_name):
        dataset = ls_client.create_dataset(dataset_name=dataset_name, description="Casos de prueba de compatibilidad RAG")
        for item in DATASET_EVALUACION:
            ls_client.create_example(inputs=item["inputs"], outputs=item["outputs"], dataset_id=dataset.id)
    
    # Corremos el experimento de evaluación en LangSmith
    results = evaluate(
        predict_rag,
        data=dataset_name,
        evaluators=[eval_context_recall, eval_faithfulness_llm_judge, eval_verdict_accuracy],
        experiment_prefix="eval-mistral-rag",
        max_concurrency=1,  # un caso a la vez, evita ráfagas de peticiones
    )
    print("\nEvaluación de LangSmith completada. Revisa los resultados en el dashboard web.")

except Exception as e:
    print(f"\nNo se pudo conectar con LangSmith ({e}). Corriendo evaluación local:\n")
    
    for item in DATASET_EVALUACION:
        q = item["inputs"]["question"]
        out = predict_rag({"question": q})
        
        class DummyObj:
            def __init__(self, d): self.outputs = d
            
        run_dummy = DummyObj(out)
        ex_dummy = DummyObj(item["outputs"])
        
        recall = eval_context_recall(run_dummy, ex_dummy)
        faith = eval_faithfulness_llm_judge(run_dummy, ex_dummy)
        acc = eval_verdict_accuracy(run_dummy, ex_dummy)
        
        print(f"Pregunta: {q}")
        print(f"  - Context Recall  : {recall['score'] * 100:.0f}%")
        print(f"  - LLM Faithfulness: {faith['score'] * 100:.0f}%")
        print(f"  - Accuracy        : {acc['score'] * 100:.0f}%\n")

=== INICIANDO EVALUACIÓN RAG Y LANGSMITH ===
View the evaluation results for experiment: 'eval-mistral-rag-7f589857' at:
https://smith.langchain.com/o/3c7b9cfa-b3b0-4abf-9ca6-9cd867fc76e8/datasets/2f2ff201-c792-4740-aa10-4ec6acd37ffe/compare?selectedSessions=cbb9a551-445e-4052-8aca-474b5b69c844


  (429 de Mistral, esperando 5s y reintentando...)
  (429 de Mistral, esperando 5s y reintentando...)
  (429 de Mistral, esperando 5s y reintentando...)
  (429 de Mistral, esperando 5s y reintentando...)
  (429 de Mistral, esperando 10s y reintentando...)
  (429 de Mistral, esperando 5s y reintentando...)

Evaluación de LangSmith completada. Revisa los resultados en el dashboard web.


# 🤖 Parte 2 (EP2): Agente funcional de SP Digital

Hasta aquí tenemos el pipeline **RAG + validador de reglas** (EP1). En esta segunda parte lo convertimos en un **agente** que decide qué herramienta usar, recuerda al cliente y planifica tareas de varias etapas.

| Indicador | Qué agregamos en este cuaderno |
|---|---|
| **IL2.1** Herramientas + framework | Agente LangChain (`create_openai_tools_agent` + `AgentExecutor`) con herramientas de **consulta**, **razonamiento** y **escritura** |
| **IL2.2** Memoria y contexto | Memoria de **corto plazo** (ventana + resumen) y de **largo plazo** persistente con **recuperación semántica** (FAISS + `mistral-embed`) |
| **IL2.3** Planificación | Planificador jerárquico con **prioridades, dependencias y replanificación reactiva** ante incompatibilidades o falta de stock |
| **IL2.4** Documentación | Diagramas de orquestación y decisiones de diseño en el `README.md` |

**Flujo general del agente**

```text
Consulta ─► [Memoria LP: recuperar contexto del cliente] ─► [Memoria CP: historial + resumen]
        ─► Agente LangChain (LLM decide herramienta)
              ├─ consulta:    consultar_catalogo · consultar_stock_precio · recordar_dato_cliente
              ├─ razonamiento: verificar_compatibilidad · buscar_alternativa · razonar_con_rag
              ├─ escritura:   generar_cotizacion · derivar_a_vendedor · guardar_en_memoria
              └─ planificación: ejecutar_plan_cotizacion ─► Planificador (prioridades + replanificación)
        ─► Respuesta ─► [Memoria CP: guarda turno] ─► [Memoria LP: guarda hechos del cliente]
```

> Todo lo anterior (catálogo, FAISS, `recuperar_contexto`, `ejecutar_pipeline_rag`, reglas y evaluación LangSmith) **se mantiene igual**: el agente lo reutiliza como herramientas.

In [ ]:
# --- Configuración del agente (IL2.1): framework LangChain + proveedor Mistral ---
import os, re, json, uuid, datetime
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

from langchain_openai import ChatOpenAI
from langchain_core.tools import StructuredTool
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor

# Mismo modelo del RAG por defecto; se puede cambiar con LLM_MODEL_AGENTE en el .env
LLM_MODEL_AGENTE = os.getenv("LLM_MODEL_AGENTE", LLM_MODEL)

# Carpeta donde el agente "escribe" (cotizaciones, derivaciones y memoria de largo plazo)
DIR_DATOS = Path("datos_agente")
DIR_COTIZACIONES = DIR_DATOS / "cotizaciones"
RUTA_MEMORIA_LP = DIR_DATOS / "memoria_largo_plazo.json"
RUTA_DERIVACIONES = DIR_DATOS / "derivaciones_vendedor.jsonl"
DIR_COTIZACIONES.mkdir(parents=True, exist_ok=True)

# Limitador de ritmo: el plan gratuito de Mistral responde 429 si se hacen muchas llamadas seguidas.
# 0.4 solicitudes/segundo = 1 llamada cada 2.5 s (el agente hace varias por turno).
limitador = InMemoryRateLimiter(requests_per_second=0.4, check_every_n_seconds=0.1, max_bucket_size=1)

# LLM del agente (LangChain). Con LANGCHAIN_TRACING_V2 activo, cada paso queda en LangSmith.
llm_agente = ChatOpenAI(
    model=LLM_MODEL_AGENTE,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0,
    max_retries=0,              # los reintentos los hace TransporteMistral (celda 1)
    http_client=HTTP_MISTRAL,
    rate_limiter=limitador,
)

# Contexto compartido por las herramientas: a qué cliente pertenece la conversación actual
CONTEXTO = {"cliente": "anonimo"}

print(f"✅ Agente configurado con {LLM_MODEL_AGENTE}. Datos en: {DIR_DATOS.resolve()}")

In [ ]:
# --- Datos adicionales para el agente (no modifican CATALOGO ni REGLAS_COMPATIBILIDAD) ---

# Inventario SIMULADO (stock y precio en CLP). RAM DDR4 queda en 0 para probar la toma de decisiones.
INVENTARIO_SIMULADO = {
    "GPU-4070-ASUS":          {"stock": 4, "precio_clp": 689990},
    "MB-B450-MSI":            {"stock": 6, "precio_clp": 94990},
    "MB-B650-ASUS":           {"stock": 3, "precio_clp": 189990},
    "CPU-R5-5600X":           {"stock": 8, "precio_clp": 119990},
    "CPU-R7-7800X3D":         {"stock": 2, "precio_clp": 379990},
    "RAM-DDR4-16GB-CORSAIR":  {"stock": 0, "precio_clp": 44990},
    "RAM-DDR5-32GB-KINGSTON": {"stock": 5, "precio_clp": 119990},
    "PSU-CORSAIR-650":        {"stock": 7, "precio_clp": 104990},
    "PSU-EVGA-450":           {"stock": 9, "precio_clp": 44990},
}

CATALOGO_POR_SKU = {p["sku"]: p for p in CATALOGO}

def _r(compatible, motivo, requisito):
    return {"compatible": compatible, "motivo": motivo, "requisito_adicional": requisito}

# Reglas duras para el agente = las 4 reglas originales + las combinaciones faltantes.
# Todas se derivan de las especificaciones del catalogo (socket, tipo de memoria, PCIe, potencia).
REGLAS_AGENTE = {
    **REGLAS_COMPATIBILIDAD,
    ("CPU-R5-5600X", "MB-B450-MSI"): _r(True, "Ambos usan socket AM4 y la B450 soporta Ryzen serie 1000 a 5000.", "Ninguno."),
    ("CPU-R5-5600X", "MB-B650-ASUS"): _r(False, "Incompatibilidad de socket: el procesador es AM4 y la placa es AM5.", "Usar una placa AM4 (ej. MSI B450) o un procesador AM5."),
    ("CPU-R7-7800X3D", "MB-B650-ASUS"): _r(True, "Ambos usan socket AM5 y la B650 soporta Ryzen 7000/8000/9000.", "El procesador requiere cooler."),
    ("RAM-DDR4-16GB-CORSAIR", "MB-B450-MSI"): _r(True, "La placa MSI B450 soporta memoria DDR4.", "Ninguno."),
    ("RAM-DDR4-16GB-CORSAIR", "MB-B650-ASUS"): _r(False, "Incompatibilidad de memoria: la placa B650 soporta solo DDR5.", "Usar memoria DDR5."),
    ("RAM-DDR5-32GB-KINGSTON", "MB-B650-ASUS"): _r(True, "La placa ASUS B650 soporta memoria DDR5.", "Ninguno."),
    ("RAM-DDR4-16GB-CORSAIR", "CPU-R5-5600X"): _r(True, "El Ryzen 5 5600X es compatible con memoria DDR4.", "Ninguno."),
    ("RAM-DDR4-16GB-CORSAIR", "CPU-R7-7800X3D"): _r(False, "El Ryzen 7 7800X3D es compatible con DDR5, no con DDR4.", "Usar memoria DDR5."),
    ("RAM-DDR5-32GB-KINGSTON", "CPU-R5-5600X"): _r(False, "El Ryzen 5 5600X es compatible con DDR4, no con DDR5.", "Usar memoria DDR4."),
    ("RAM-DDR5-32GB-KINGSTON", "CPU-R7-7800X3D"): _r(True, "El Ryzen 7 7800X3D es compatible con memoria DDR5.", "Ninguno."),
    ("GPU-4070-ASUS", "MB-B650-ASUS"): _r(True, "El slot PCIe 4.0 x16 de la placa coincide con la GPU.", "Fuente >= 650W con conector 12VHPWR o adaptador."),
    ("GPU-4070-ASUS", "PSU-CORSAIR-650"): _r(True, "La fuente de 650W cumple el minimo y trae conector 12VHPWR (con adaptador).", "Ninguno."),
}

# Pares de categorias que SI requieren verificacion de compatibilidad
PARES_RELEVANTES = {
    frozenset(("Procesador", "Placa madre")),
    frozenset(("Memoria RAM", "Placa madre")),
    frozenset(("Memoria RAM", "Procesador")),
    frozenset(("GPU", "Placa madre")),
    frozenset(("GPU", "Fuente de poder")),
}

# Alias para reconocer productos escritos en lenguaje natural
ALIAS_SKU = {
    "GPU-4070-ASUS": ["4070", "rtx"],
    "MB-B450-MSI": ["b450", "tomahawk"],
    "MB-B650-ASUS": ["b650"],
    "CPU-R5-5600X": ["5600x", "ryzen 5"],
    "CPU-R7-7800X3D": ["7800x3d", "ryzen 7"],
    "RAM-DDR4-16GB-CORSAIR": ["ddr4", "vengeance"],
    "RAM-DDR5-32GB-KINGSTON": ["ddr5", "fury beast", "kingston"],
    "PSU-CORSAIR-650": ["rm650", "corsair 650", "650w"],
    "PSU-EVGA-450": ["evga", "450w"],
}

def buscar_regla(sku_a, sku_b):
    """Busca la regla del par sin importar el orden."""
    return REGLAS_AGENTE.get((sku_a, sku_b)) or REGLAS_AGENTE.get((sku_b, sku_a))

def es_par_relevante(sku_a, sku_b):
    if sku_a not in CATALOGO_POR_SKU or sku_b not in CATALOGO_POR_SKU:
        return False
    return frozenset((CATALOGO_POR_SKU[sku_a]["categoria"], CATALOGO_POR_SKU[sku_b]["categoria"])) in PARES_RELEVANTES

def extraer_skus(texto):
    """Devuelve los SKUs mencionados en un texto (por SKU exacto o alias), sin repetir."""
    t = texto.lower()
    return [sku for sku in CATALOGO_POR_SKU
            if sku.lower() in t or any(a in t for a in ALIAS_SKU.get(sku, []))]

def resolver_sku(texto):
    """Acepta un SKU exacto o un nombre en lenguaje natural. Devuelve None si no es unico."""
    limpio = texto.strip().upper()
    if limpio in CATALOGO_POR_SKU:
        return limpio
    encontrados = extraer_skus(texto)
    return encontrados[0] if len(encontrados) == 1 else None

def evaluar_par(sku_a, sku_b):
    """Aplica el validador de reglas a un par de SKUs (misma logica que el pipeline RAG)."""
    regla = buscar_regla(sku_a, sku_b)
    if not regla:
        return {"veredicto": "SIN REGLA -> DERIVAR", "sku_a": sku_a, "sku_b": sku_b,
                "motivo": "No existe una regla de compatibilidad registrada para este par.",
                "requisito_adicional": None}
    return {"veredicto": "COMPATIBLE" if regla["compatible"] else "NO COMPATIBLE",
            "sku_a": sku_a, "sku_b": sku_b,
            "motivo": regla["motivo"], "requisito_adicional": regla["requisito_adicional"]}

print(f"Reglas del agente: {len(REGLAS_AGENTE)} (originales: {len(REGLAS_COMPATIBILIDAD)}) | Productos con inventario: {len(INVENTARIO_SIMULADO)}")

## 🧠 IL2.2 · Memoria y recuperación de contexto

El agente usa **dos memorias**, siguiendo las estrategias vistas en RA2 (Buffer / Window / Summary):

| Memoria | Qué guarda | Cómo se mantiene | Cuándo se usa |
|---|---|---|---|
| **Corto plazo** (`MemoriaCortoPlazo`) | Últimos *k* turnos de la conversación + un **resumen acumulado** de los turnos más antiguos | Ventana deslizante; al salir un turno de la ventana, el LLM lo integra al resumen | Preguntas de seguimiento dentro de la misma sesión |
| **Largo plazo** (`MemoriaLargoPlazo`) | Hechos duraderos del cliente (equipo que posee, presupuesto, uso) | Archivo JSON persistente con el **embedding** de cada hecho, aislado por cliente | Se recupera **semánticamente** (FAISS + `mistral-embed`) en cada consulta, incluso en una sesión nueva |

La ventana evita que el prompt crezca sin límite y el resumen evita perder datos de turnos antiguos (*flujos prolongados*).

In [ ]:
# --- Memoria de corto plazo (ventana + resumen) y de largo plazo (persistente + semántica) ---

def resumir_con_llm(resumen_previo, usuario, asistente):
    """Integra un turno antiguo al resumen acumulado (equivalente a ConversationSummaryMemory)."""
    prompt = (
        "Actualiza el resumen de una conversacion de soporte tecnico de hardware.\n"
        "Conserva SKUs, equipos del cliente, decisiones tomadas y datos pendientes. "
        "No inventes nada. Maximo 80 palabras.\n\n"
        f"Resumen actual: {resumen_previo or '(vacio)'}\n"
        f"Turno a integrar -> Cliente: {usuario}\nAsistente: {asistente}\n\n"
        "Nuevo resumen:"
    )
    resp = llamar_con_espera(llm_agente.invoke, prompt)
    return resp.content.strip()


class MemoriaCortoPlazo:
    """Ventana de los ultimos k turnos + resumen acumulado de los anteriores."""

    def __init__(self, resumidor: Optional[Callable] = resumir_con_llm, k: int = 3):
        self.k = k
        self.resumidor = resumidor
        self.turnos: List[tuple] = []      # [(usuario, asistente), ...]
        self.resumen: str = ""

    def agregar_turno(self, usuario: str, asistente: str):
        self.turnos.append((usuario, asistente))
        while len(self.turnos) > self.k:
            u, a = self.turnos.pop(0)
            self.resumen = self._resumir(u, a)

    def _resumir(self, u, a):
        if self.resumidor:
            try:
                return self.resumidor(self.resumen, u, a)
            except Exception:
                pass  # si el LLM falla, usamos el resumen simple de respaldo
        simple = f"{self.resumen} | Cliente: {u[:120]} -> Asistente: {a[:120]}".strip(" |")
        return simple[-800:]

    def historial_mensajes(self):
        """Historial en formato que entiende MessagesPlaceholder de LangChain."""
        msgs = []
        for u, a in self.turnos:
            msgs += [("human", u), ("ai", a)]
        return msgs

    def reiniciar(self):
        self.turnos, self.resumen = [], ""


class MemoriaLargoPlazo:
    """Hechos duraderos por cliente, guardados en disco y recuperados por similitud semantica."""

    def __init__(self, ruta, embed_fn: Callable):
        self.ruta = Path(ruta)
        self.embed_fn = embed_fn
        self.entradas: List[dict] = []
        self._cargar()

    def _cargar(self):
        if self.ruta.exists():
            try:
                self.entradas = json.loads(self.ruta.read_text(encoding="utf-8"))
            except (json.JSONDecodeError, OSError):
                self.entradas = []

    def _guardar(self):
        self.ruta.parent.mkdir(parents=True, exist_ok=True)
        self.ruta.write_text(json.dumps(self.entradas, ensure_ascii=False), encoding="utf-8")

    def _vector(self, texto):
        v = np.array(self.embed_fn([texto]), dtype="float32")
        faiss.normalize_L2(v)
        return v

    def agregar(self, texto: str, cliente: str = "anonimo", tipo: str = "hecho") -> str:
        texto = texto.strip()
        for e in self.entradas:                       # evita duplicados exactos
            if e["cliente"] == cliente and e["texto"].lower() == texto.lower():
                return e["id"]
        entrada = {
            "id": uuid.uuid4().hex[:8], "cliente": cliente, "tipo": tipo, "texto": texto,
            "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
            "embedding": self._vector(texto)[0].tolist(),
        }
        self.entradas.append(entrada)
        self._guardar()
        return entrada["id"]

    def recuperar(self, consulta: str, k: int = 3, cliente: Optional[str] = None, umbral: float = 0.0):
        """Los k hechos mas cercanos semanticamente a la consulta (solo del cliente indicado)."""
        pos = [i for i, e in enumerate(self.entradas) if cliente is None or e["cliente"] == cliente]
        if not pos:
            return []
        mat = np.array([self.entradas[i]["embedding"] for i in pos], dtype="float32")
        faiss.normalize_L2(mat)
        idx = faiss.IndexFlatIP(mat.shape[1])
        idx.add(mat)
        scores, cand = idx.search(self._vector(consulta), min(k, len(pos)))
        salida = []
        for s, c in zip(scores[0], cand[0]):
            if c != -1 and s >= umbral:
                e = self.entradas[pos[c]]
                salida.append({kk: vv for kk, vv in e.items() if kk != "embedding"} | {"score": float(s)})
        return salida

    def hechos_de(self, cliente: str):
        return [e["texto"] for e in self.entradas if e["cliente"] == cliente]

    def vaciar(self):
        self.entradas = []
        self._guardar()


def _embed_con_pausa(textos):
    time.sleep(1.5)                      # evita ráfagas de peticiones a Mistral
    return llamar_con_espera(obtener_embeddings, textos)

memoria_lp = MemoriaLargoPlazo(RUTA_MEMORIA_LP, embed_fn=_embed_con_pausa)
print(f"✅ Memoria de largo plazo lista ({len(memoria_lp.entradas)} hechos guardados en {RUTA_MEMORIA_LP}).")

## 🛠️ IL2.1 · Herramientas del agente

Cada herramienta es una función Python con *docstring* (el LLM la lee para decidir cuándo usarla) y **nunca lanza excepciones**: devuelve el error como texto para que el agente pueda reaccionar (buena práctica vista en RA2).

| Tipo | Herramienta | Qué hace |
|---|---|---|
| 🔎 Consulta | `consultar_catalogo` | Búsqueda vectorial FAISS (reutiliza `recuperar_contexto`) |
| 🔎 Consulta | `consultar_stock_precio` | Existencia, stock y precio de un SKU |
| 🔎 Consulta | `recordar_dato_cliente` | Recuperación semántica en la memoria de largo plazo |
| 🧮 Razonamiento | `verificar_compatibilidad` | Validador de reglas duras para un par de SKUs |
| 🧮 Razonamiento | `buscar_alternativa` | Busca un reemplazo en stock compatible con el resto de la configuración |
| 🧮 Razonamiento | `razonar_con_rag` | Ejecuta el pipeline RAG completo de la EP1 (`ejecutar_pipeline_rag`) |
| ✍️ Escritura | `generar_cotizacion` | Valida y escribe un archivo `COT-*.json` (rechaza configuraciones inválidas) |
| ✍️ Escritura | `derivar_a_vendedor` | Registra un ticket de derivación en `derivaciones_vendedor.jsonl` |
| ✍️ Escritura | `guardar_en_memoria` | Guarda un hecho duradero del cliente en la memoria de largo plazo |

In [ ]:
# --- Herramientas del agente (funciones puras; se registran en LangChain más abajo) ---

def _json(d):
    return json.dumps(d, ensure_ascii=False)

def _ahora():
    return datetime.datetime.now().isoformat(timespec="seconds")

# ---------- CONSULTA ----------
def consultar_catalogo(pregunta: str) -> str:
    """Busca en el catalogo vectorial (FAISS) los productos y notas tecnicas mas relevantes para una pregunta y devuelve su SKU y especificaciones. Usala para conocer las especificaciones de un producto."""
    try:
        fragmentos = llamar_con_espera(recuperar_contexto, pregunta, top_k=3)
        return "\n".join(f"[{f['metadata']['sku']}] {f['texto']}" for f in fragmentos) or "Sin resultados en el catalogo."
    except Exception as e:
        return f"ERROR consultando el catalogo: {e}"

def consultar_stock_precio(sku: str) -> str:
    """Consulta si un producto existe en el catalogo y devuelve su stock y precio en CLP. Recibe un SKU (ej. GPU-4070-ASUS) o el nombre del producto."""
    try:
        s = resolver_sku(sku)
        if not s:
            return _json({"sku": sku, "existe": False, "disponible": False,
                          "mensaje": "El producto no existe en el catalogo de SP Digital."})
        inv = INVENTARIO_SIMULADO.get(s, {"stock": 0, "precio_clp": None})
        return _json({"sku": s, "nombre": CATALOGO_POR_SKU[s]["nombre"], "existe": True,
                      "stock": inv["stock"], "precio_clp": inv["precio_clp"], "disponible": inv["stock"] > 0})
    except Exception as e:
        return _json({"sku": sku, "existe": False, "disponible": False, "error": str(e)})

def recordar_dato_cliente(consulta: str) -> str:
    """Busca en la memoria de largo plazo datos previamente guardados sobre el cliente actual (equipo que posee, presupuesto, preferencias) relacionados con la consulta."""
    try:
        r = memoria_lp.recuperar(consulta, k=4, cliente=CONTEXTO["cliente"])
        return "\n".join(f"- {x['texto']}" for x in r) or "No hay datos previos de este cliente."
    except Exception as e:
        return f"ERROR leyendo la memoria: {e}"

# ---------- RAZONAMIENTO ----------
def verificar_compatibilidad(sku_a: str, sku_b: str) -> str:
    """Verifica con las reglas de negocio si dos productos del catalogo son compatibles entre si. Devuelve COMPATIBLE, NO COMPATIBLE o SIN REGLA -> DERIVAR, con el motivo. Recibe dos SKUs o nombres de producto."""
    try:
        a, b = resolver_sku(sku_a), resolver_sku(sku_b)
        if not a or not b:
            return _json({"veredicto": "SIN REGLA -> DERIVAR",
                          "motivo": "Alguno de los productos no existe en el catalogo."})
        return _json(evaluar_par(a, b))
    except Exception as e:
        return _json({"veredicto": "SIN REGLA -> DERIVAR", "motivo": f"Error: {e}"})

def buscar_alternativa(sku_a_reemplazar: str, skus_restantes: str) -> str:
    """Busca productos de la misma categoria que sku_a_reemplazar, con stock y compatibles con el resto de la configuracion. skus_restantes es una lista de SKUs separados por coma. Devuelve candidatos ordenados por precio."""
    try:
        objetivo = resolver_sku(sku_a_reemplazar)
        if not objetivo:
            return _json({"candidatos": [], "mensaje": "SKU a reemplazar no reconocido."})
        categoria = CATALOGO_POR_SKU[objetivo]["categoria"]
        resto = [resolver_sku(x) for x in skus_restantes.split(",") if x.strip()]
        resto = [r for r in resto if r and CATALOGO_POR_SKU[r]["categoria"] != categoria]
        candidatos = []
        for p in CATALOGO:
            sku = p["sku"]
            inv = INVENTARIO_SIMULADO.get(sku, {"stock": 0, "precio_clp": None})
            if p["categoria"] != categoria or sku == objetivo or inv["stock"] <= 0:
                continue
            pares = [evaluar_par(sku, r)["veredicto"] for r in resto if es_par_relevante(sku, r)]
            if all(v == "COMPATIBLE" for v in pares):
                candidatos.append({"sku": sku, "nombre": p["nombre"], "precio_clp": inv["precio_clp"], "stock": inv["stock"]})
        candidatos.sort(key=lambda c: c["precio_clp"] or 0)
        return _json({"reemplaza": objetivo, "candidatos": candidatos})
    except Exception as e:
        return _json({"candidatos": [], "error": str(e)})

def razonar_con_rag(pregunta: str) -> str:
    """Responde una pregunta tecnica de compatibilidad ejecutando el pipeline RAG completo (busqueda vectorial + LLM + validador de reglas). Devuelve la explicacion y el veredicto del validador."""
    try:
        res = ejecutar_pipeline_rag(pregunta)
        regla = res["regla_aplicada"]
        veredicto = ("COMPATIBLE" if regla["compatible"] else "NO COMPATIBLE") if regla else "SIN REGLA -> DERIVAR"
        return _json({"veredicto_validador": veredicto, "skus_recuperados": res["skus_recuperados"],
                      "explicacion_rag": res["respuesta_llm"][:700]})
    except Exception as e:
        return _json({"veredicto_validador": "SIN REGLA -> DERIVAR", "error": str(e)})

# ---------- ESCRITURA ----------
def generar_cotizacion(skus: str) -> str:
    """Genera y guarda una cotizacion (archivo JSON) para una lista de SKUs separados por coma. Valida existencia, stock y compatibilidad de todos los pares; si hay un problema NO emite la cotizacion y lo informa."""
    try:
        lista = []
        for x in skus.split(","):
            if x.strip():
                s = resolver_sku(x)
                if not s:
                    return _json({"estado": "RECHAZADA", "problemas": [f"Producto no reconocido: {x.strip()}"]})
                if s not in lista:
                    lista.append(s)
        if not lista:
            return _json({"estado": "RECHAZADA", "problemas": ["No se indicaron productos."]})

        problemas, advertencias = [], []
        for s in lista:
            if INVENTARIO_SIMULADO.get(s, {"stock": 0})["stock"] <= 0:
                problemas.append(f"Sin stock: {s}")
        for i, a in enumerate(lista):
            for b in lista[i + 1:]:
                if es_par_relevante(a, b):
                    ev = evaluar_par(a, b)
                    if ev["veredicto"] != "COMPATIBLE":
                        problemas.append(f"{a} <-> {b}: {ev['veredicto']} ({ev['motivo']})")
        cats = {CATALOGO_POR_SKU[s]["categoria"] for s in lista}
        if "GPU" in cats and "Fuente de poder" not in cats:
            advertencias.append("La cotizacion incluye GPU sin fuente: verificar fuente de 650W o mas con 12VHPWR.")
        if problemas:
            return _json({"estado": "RECHAZADA", "problemas": problemas})

        items = [{"sku": s, "nombre": CATALOGO_POR_SKU[s]["nombre"], "precio_clp": INVENTARIO_SIMULADO[s]["precio_clp"]} for s in lista]
        cot_id = f"COT-{datetime.date.today():%Y%m%d}-{uuid.uuid4().hex[:4].upper()}"
        doc = {"id": cot_id, "cliente": CONTEXTO["cliente"], "fecha": _ahora(), "items": items,
               "total_clp": sum(i["precio_clp"] for i in items), "advertencias": advertencias,
               "nota": "Precios y stock simulados (prototipo academico)."}
        DIR_COTIZACIONES.mkdir(parents=True, exist_ok=True)
        ruta = DIR_COTIZACIONES / f"{cot_id}.json"
        ruta.write_text(json.dumps(doc, ensure_ascii=False, indent=2), encoding="utf-8")
        return _json({"estado": "EMITIDA", "id": cot_id, "ruta": str(ruta), "total_clp": doc["total_clp"], "advertencias": advertencias})
    except Exception as e:
        return _json({"estado": "RECHAZADA", "problemas": [f"Error interno: {e}"]})

def derivar_a_vendedor(motivo: str, consulta: str) -> str:
    """Deriva el caso a un vendedor humano registrando un ticket. Usala cuando falte informacion, el producto no exista en el catalogo, no haya regla de compatibilidad, no haya stock ni alternativa, o el cliente pida algo fuera del catalogo."""
    try:
        ticket = {"ticket": f"DER-{uuid.uuid4().hex[:6].upper()}", "fecha": _ahora(),
                  "cliente": CONTEXTO["cliente"], "motivo": motivo, "consulta": consulta}
        RUTA_DERIVACIONES.parent.mkdir(parents=True, exist_ok=True)
        with open(RUTA_DERIVACIONES, "a", encoding="utf-8") as f:
            f.write(json.dumps(ticket, ensure_ascii=False) + "\n")
        return _json({"estado": "DERIVADO", "ticket": ticket["ticket"], "motivo": motivo})
    except Exception as e:
        return _json({"estado": "ERROR", "error": str(e)})

def guardar_en_memoria(dato: str) -> str:
    """Guarda en la memoria de largo plazo un dato duradero del cliente actual (equipo que ya posee, presupuesto, uso que le dara al PC). Usala cuando el cliente declare algo que convenga recordar en futuras conversaciones."""
    try:
        mid = memoria_lp.agregar(dato, cliente=CONTEXTO["cliente"], tipo="hecho")
        return _json({"estado": "GUARDADO", "id": mid, "dato": dato})
    except Exception as e:
        return _json({"estado": "ERROR", "error": str(e)})

print("✅ 9 herramientas definidas (consulta, razonamiento y escritura).")

## 🗓️ IL2.3 · Planificación y toma de decisiones

Para pedidos de **varias etapas** (cotizar o armar un PC completo) el agente delega en un **planificador** con tres ideas del RA2:

1. **Jerárquico**: el objetivo se descompone en subtareas (`PlanStep`) con **prioridad** (1 = alta … 3 = baja) y **dependencias**.
2. **Priorizado**: en cada ciclo se ejecuta primero el paso listo con mayor prioridad (diagnóstico antes que cotizar).
3. **Reactivo (replanificación)**: si un resultado cambia las condiciones (incompatibilidad, sin stock) el plan se **modifica en ejecución**:

```text
Objetivo ─► Plan v1: [stock × N] ─► [compatibilidad × pares] (P1) ──► ¿problemas?
                                                                    ├─ no ─► generar_cotizacion (P3)
                                                                    └─ sí ─► elegir componente culpable
                                                                              ─► buscar_alternativa (P1)
                                                                                 ├─ hay ─► Plan v2 (sustituye y re-verifica) ─► ciclo
                                                                                 └─ no  ─► derivar_a_vendedor (P1)
```

Regla de decisión: se reemplaza el componente involucrado en **más conflictos** (incompatibilidades o falta de stock); en caso de empate, el más fácil de cambiar (placa → fuente → RAM → procesador → GPU). Máximo 3 replanificaciones.

In [ ]:
# --- Planificador con prioridades, dependencias y replanificación reactiva ---
from langsmith import traceable

# Registro de herramientas que el planificador puede invocar (sin LLM: es determinista y auditable)
HERRAMIENTAS_FN = {
    "consultar_stock_precio": consultar_stock_precio,
    "verificar_compatibilidad": verificar_compatibilidad,
    "buscar_alternativa": buscar_alternativa,
    "generar_cotizacion": generar_cotizacion,
    "derivar_a_vendedor": derivar_a_vendedor,
}

@traceable(name="Planner_Tool_Call", run_type="tool")
def invocar_herramienta(nombre: str, args: dict) -> str:
    """Ejecuta una herramienta por nombre (queda trazada en LangSmith)."""
    return HERRAMIENTAS_FN[nombre](**args)

PRIORIDAD_REEMPLAZO = ["Placa madre", "Fuente de poder", "Memoria RAM", "Procesador", "GPU"]
MAX_REPLANIFICACIONES = 3
NOMBRE_PRIORIDAD = {1: "ALTA", 2: "MEDIA", 3: "BAJA"}


@dataclass
class PlanStep:
    id: int
    accion: str                      # nombre de la herramienta
    descripcion: str
    args: Dict[str, Any]
    prioridad: int = 2               # 1 = alta, 3 = baja
    dependencias: List[int] = field(default_factory=list)
    estado: str = "pendiente"        # pendiente | completado | fallido | omitido
    resultado: Any = None
    ronda: int = 0                   # 0 = plan original, >0 = agregado al replanificar


@dataclass
class Plan:
    objetivo: str
    skus: List[str]
    pasos: List[PlanStep] = field(default_factory=list)
    sustituciones: List[dict] = field(default_factory=list)
    bitacora: List[str] = field(default_factory=list)
    version: int = 1
    estado: str = "creado"           # creado | COTIZACION_EMITIDA | DERIVADO


def _parse(texto):
    try:
        return json.loads(texto)
    except (TypeError, json.JSONDecodeError):
        return {"raw": texto}


class PlanificadorCotizacion:
    def __init__(self, ejecutor: Callable = invocar_herramienta, verbose: bool = True):
        self.ejecutor = ejecutor
        self.verbose = verbose

    # ---------- utilidades ----------
    def _log(self, plan, msg):
        plan.bitacora.append(msg)
        if self.verbose:
            print(msg)

    def _nuevo_paso(self, plan, accion, descripcion, args, prioridad=2, dependencias=None, ronda=0):
        paso = PlanStep(id=len(plan.pasos) + 1, accion=accion, descripcion=descripcion, args=args,
                        prioridad=prioridad, dependencias=dependencias or [], ronda=ronda)
        plan.pasos.append(paso)
        return paso

    def _id_stock(self, plan, sku):
        for p in reversed(plan.pasos):
            if p.accion == "consultar_stock_precio" and p.args["sku"] == sku and p.estado != "omitido":
                return p.id
        return None

    # ---------- 1. planificación jerárquica ----------
    def crear_plan(self, objetivo: str) -> Plan:
        return Plan(objetivo=objetivo, skus=extraer_skus(objetivo))

    def _agregar_diagnostico(self, plan, skus_nuevos, ronda):
        for s in skus_nuevos:
            self._nuevo_paso(plan, "consultar_stock_precio", f"Verificar existencia y stock de {s}",
                             {"sku": s}, prioridad=1, ronda=ronda)
        for i, a in enumerate(plan.skus):
            for b in plan.skus[i + 1:]:
                if (a in skus_nuevos or b in skus_nuevos) and es_par_relevante(a, b):
                    deps = [x for x in (self._id_stock(plan, a), self._id_stock(plan, b)) if x]
                    self._nuevo_paso(plan, "verificar_compatibilidad", f"Verificar compatibilidad {a} <-> {b}",
                                     {"sku_a": a, "sku_b": b}, prioridad=1, dependencias=deps, ronda=ronda)

    # ---------- 2. ejecución priorizada (respeta dependencias) ----------
    def _ejecutar_paso(self, plan, p):
        self._log(plan, f"   [prioridad {NOMBRE_PRIORIDAD[p.prioridad]}] paso {p.id}: {p.descripcion}")
        try:
            p.resultado = _parse(self.ejecutor(p.accion, p.args))
            p.estado = "completado"
        except Exception as e:
            p.resultado, p.estado = {"error": str(e)}, "fallido"
        r = p.resultado
        resumen = r.get("veredicto") or r.get("estado") or (
            f"stock={r.get('stock')}" if "stock" in r else (f"{len(r['candidatos'])} candidato(s)" if "candidatos" in r else str(r)[:80]))
        self._log(plan, f"        ↳ {resumen}")

    def _ejecutar_pendientes(self, plan):
        while True:
            listos = [p for p in plan.pasos if p.estado == "pendiente" and all(
                plan.pasos[d - 1].estado in ("completado", "omitido", "fallido") for d in p.dependencias)]
            if not listos:
                break
            self._ejecutar_paso(plan, min(listos, key=lambda x: (x.prioridad, x.id)))

    # ---------- 3. diagnóstico y replanificación reactiva ----------
    def _diagnosticar(self, plan):
        pr = {"inexistentes": [], "sin_stock": [], "incompatibles": [], "sin_regla": []}
        for p in plan.pasos:
            if p.estado == "omitido":
                continue
            r = p.resultado or {}
            if p.estado == "fallido":
                pr["sin_regla"].append(p.descripcion)
            elif p.accion == "consultar_stock_precio":
                if not r.get("existe", False):
                    pr["inexistentes"].append(p.args["sku"])
                elif not r.get("disponible", False):
                    pr["sin_stock"].append(p.args["sku"])
            elif p.accion == "verificar_compatibilidad":
                if r.get("veredicto") == "NO COMPATIBLE":
                    pr["incompatibles"].append((p.args["sku_a"], p.args["sku_b"], r.get("motivo", "")))
                elif r.get("veredicto") != "COMPATIBLE":
                    pr["sin_regla"].append(f"{p.args['sku_a']} <-> {p.args['sku_b']}")
        return pr

    @staticmethod
    def _hay_problemas(pr):
        return any(pr.values())

    @staticmethod
    def _texto_problemas(pr):
        partes = []
        if pr["inexistentes"]: partes.append(f"productos fuera de catalogo: {pr['inexistentes']}")
        if pr["sin_stock"]: partes.append(f"sin stock: {pr['sin_stock']}")
        if pr["incompatibles"]: partes.append("incompatibilidades: " + "; ".join(f"{a} <-> {b}" for a, b, _ in pr["incompatibles"]))
        if pr["sin_regla"]: partes.append(f"sin regla de compatibilidad: {pr['sin_regla']}")
        return " | ".join(partes)

    def _elegir_culpable(self, plan, pr):
        conteo = {}
        for a, b, _ in pr["incompatibles"]:
            conteo[a] = conteo.get(a, 0) + 1
            conteo[b] = conteo.get(b, 0) + 1
        for s in pr["sin_stock"]:
            conteo[s] = conteo.get(s, 0) + 1
        return max(conteo, key=lambda s: (conteo[s], -PRIORIDAD_REEMPLAZO.index(CATALOGO_POR_SKU[s]["categoria"])))

    def _replanificar(self, plan, pr, ronda) -> bool:
        """Modifica el plan. Devuelve False si no hay forma automática de resolverlo."""
        if pr["inexistentes"] or pr["sin_regla"]:
            self._log(plan, "   🧭 Decisión: falta información/regla para validar → derivar a vendedor humano.")
            return False
        culpable = self._elegir_culpable(plan, pr)
        resto = [s for s in plan.skus if s != culpable]
        self._log(plan, f"   🧭 Decisión: reemplazar {culpable} (componente más problemático). Replanificando (plan v{plan.version + 1})...")
        paso = self._nuevo_paso(plan, "buscar_alternativa", f"Buscar alternativa en stock para {culpable}",
                                {"sku_a_reemplazar": culpable, "skus_restantes": ",".join(resto)}, prioridad=1, ronda=ronda)
        self._ejecutar_paso(plan, paso)
        candidatos = (paso.resultado or {}).get("candidatos", [])
        if not candidatos:
            self._log(plan, f"   🧭 Decisión: no hay alternativa en catálogo para {culpable} → derivar a vendedor humano.")
            return False
        nuevo = candidatos[0]["sku"]            # el más barato con stock y compatible
        plan.skus = [nuevo if s == culpable else s for s in plan.skus]
        for p in plan.pasos:                    # los pasos del componente descartado ya no aplican
            if p.estado == "completado" and culpable in (p.args.get("sku"), p.args.get("sku_a"), p.args.get("sku_b")):
                p.estado = "omitido"
        plan.sustituciones.append({"reemplaza": culpable, "por": nuevo, "ronda": ronda})
        plan.version += 1
        self._agregar_diagnostico(plan, [nuevo], ronda)
        return True

    # ---------- ciclo completo ----------
    def ejecutar(self, objetivo: str) -> dict:
        plan = self.crear_plan(objetivo)
        self._log(plan, f"🎯 Objetivo: {objetivo}")
        pr = {"inexistentes": [], "sin_stock": [], "incompatibles": [], "sin_regla": []}
        if not plan.skus:
            pr["inexistentes"] = ["(no se identificaron productos del catálogo en la solicitud)"]
            return self._derivar(plan, pr)

        self._log(plan, f"📋 Plan v1 para {plan.skus}")
        self._agregar_diagnostico(plan, list(plan.skus), 0)
        resuelto = False
        for ronda in range(MAX_REPLANIFICACIONES + 1):
            self._ejecutar_pendientes(plan)
            pr = self._diagnosticar(plan)
            if not self._hay_problemas(pr):
                resuelto = True
                break
            self._log(plan, f"   ⚠️ Problemas detectados: {self._texto_problemas(pr)}")
            if ronda == MAX_REPLANIFICACIONES or not self._replanificar(plan, pr, ronda + 1):
                break

        if resuelto:
            activos = [p.id for p in plan.pasos if p.estado != "omitido"]
            paso = self._nuevo_paso(plan, "generar_cotizacion", f"Emitir cotización para {plan.skus}",
                                    {"skus": ",".join(plan.skus)}, prioridad=3, dependencias=activos, ronda=ronda)
            self._ejecutar_pendientes(plan)
            if (paso.resultado or {}).get("estado") == "EMITIDA":
                plan.estado = "COTIZACION_EMITIDA"
                self._log(plan, f"✅ Cotización emitida: {paso.resultado['id']} (total ${paso.resultado['total_clp']:,} CLP)")
                return self._resultado(plan, paso.resultado)
            pr["sin_regla"].append(f"cotización rechazada: {(paso.resultado or {}).get('problemas')}")
        return self._derivar(plan, pr)

    def _derivar(self, plan, pr):
        motivo = self._texto_problemas(pr) or "No fue posible completar la solicitud automáticamente."
        paso = self._nuevo_paso(plan, "derivar_a_vendedor", "Derivar caso a vendedor humano",
                                {"motivo": motivo, "consulta": plan.objetivo}, prioridad=1)
        self._ejecutar_pendientes(plan)
        plan.estado = "DERIVADO"
        self._log(plan, f"📨 Derivado a vendedor humano: {(paso.resultado or {}).get('ticket')} — {motivo}")
        return self._resultado(plan, paso.resultado)

    def _resultado(self, plan, final):
        return {"estado": plan.estado, "skus_finales": plan.skus, "sustituciones": plan.sustituciones,
                "version_plan": plan.version, "resultado_final": final, "bitacora": plan.bitacora, "plan": plan}


def mostrar_plan(plan: Plan):
    """Tabla del plan: orden, prioridad, dependencias y estado final de cada paso."""
    print(f"{'#':<4}{'Prior.':<8}{'Ronda':<7}{'Deps':<20}{'Estado':<12}Acción")
    for p in plan.pasos:
        deps = str(p.dependencias) if len(p.dependencias) <= 4 else f"[{len(p.dependencias)} pasos]"
        print(f"{p.id:<4}{NOMBRE_PRIORIDAD[p.prioridad]:<8}{p.ronda:<7}{deps:<20}{p.estado:<12}{p.accion}: {p.descripcion}")


planificador = PlanificadorCotizacion()

@traceable(name="Plan_Cotizacion_SP_Digital", run_type="chain")
def planificar_y_ejecutar(objetivo: str) -> dict:
    res = planificador.ejecutar(objetivo)
    return {k: v for k, v in res.items() if k != "plan"}     # el Plan completo no es serializable en la traza

def ejecutar_plan_cotizacion(objetivo: str) -> str:
    """Planifica y ejecuta automaticamente una cotizacion o armado de PC con DOS o mas componentes: verifica stock y compatibilidad por prioridades, reemplaza componentes si hay incompatibilidad o falta de stock, y emite la cotizacion o deriva a un vendedor. Recibe el pedido del cliente en lenguaje natural, mencionando los productos."""
    try:
        res = planificar_y_ejecutar(objetivo)
        return _json({"estado": res["estado"], "skus_finales": res["skus_finales"],
                      "sustituciones": res["sustituciones"], "resultado_final": res["resultado_final"]})
    except Exception as e:
        return _json({"estado": "ERROR", "error": str(e)})

print("✅ Planificador listo (prioridades + dependencias + replanificación reactiva).")

In [ ]:
# --- Agente LangChain: herramientas + memoria + planificador, orquestados ---

TOOLS_AGENTE = [StructuredTool.from_function(f) for f in (
    consultar_catalogo, consultar_stock_precio, recordar_dato_cliente,        # consulta
    verificar_compatibilidad, buscar_alternativa, razonar_con_rag,            # razonamiento
    generar_cotizacion, derivar_a_vendedor, guardar_en_memoria,               # escritura
    ejecutar_plan_cotizacion,                                                 # planificación
)]

SYSTEM_PROMPT_AGENTE = """Eres el agente de ventas y soporte tecnico de SP Digital (componentes de PC).
REGLAS:
1. Para datos de catalogo, stock, precios o compatibilidad usa SIEMPRE las herramientas. Nunca inventes especificaciones, precios, stock ni modelos.
2. Compatibilidad entre dos productos: verificar_compatibilidad. Si el cliente pide una explicacion tecnica detallada, razonar_con_rag.
3. Cotizar o armar un PC con dos o mas componentes: ejecutar_plan_cotizacion con el pedido completo del cliente.
4. Si una herramienta indica SIN REGLA, producto inexistente, sin stock sin alternativa, o el pedido esta fuera del catalogo, usa derivar_a_vendedor y avisa al cliente.
5. Si el cliente declara algo duradero (equipo que posee, presupuesto, uso del PC), guardalo con guardar_en_memoria.
6. Usa la memoria de largo plazo y el resumen de la conversacion para no volver a preguntar datos que ya conoces.
7. Responde en espanol, tono tecnico y breve (maximo 120 palabras). En consultas de compatibilidad usa: Veredicto / Justificacion / Recomendacion.

<memoria_largo_plazo>
{memoria_largo}
</memoria_largo_plazo>

<resumen_conversacion>
{resumen}
</resumen_conversacion>"""

PROMPT_AGENTE = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_AGENTE),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

_agente_base = create_openai_tools_agent(llm_agente, TOOLS_AGENTE, PROMPT_AGENTE)

# Respaldo determinista: frases declarativas del cliente que conviene recordar
PATRON_HECHO = re.compile(r"\b(tengo|poseo|presupuesto|juego|jugar|trabajo con)\b", re.IGNORECASE)


class AgenteSPDigital:
    """Orquestador: memoria de largo plazo -> memoria de corto plazo -> agente con herramientas -> actualizar memorias."""

    def __init__(self, cliente: str = "anonimo", k_ventana: int = 3, memoria_largo=None, verbose: bool = True):
        self.cliente = cliente
        self.lp = memoria_largo or memoria_lp
        self.cp = MemoriaCortoPlazo(k=k_ventana)       # memoria de corto plazo nueva por sesión
        self.executor = AgentExecutor(agent=_agente_base, tools=TOOLS_AGENTE, verbose=verbose,
                                      max_iterations=4, handle_parsing_errors=True,
                                      return_intermediate_steps=True)

    def _memorizar_declaraciones(self, consulta, herramientas):
        if "guardar_en_memoria" in herramientas:         # el agente ya la guardó con su herramienta
            return []
        guardados = []
        for frase in re.split(r"(?<=[.!])\s+", consulta):
            if "?" in frase or "¿" in frase:
                continue
            if PATRON_HECHO.search(frase):
                self.lp.agregar(frase.strip(), cliente=self.cliente, tipo="declaracion")
                guardados.append(frase.strip())
        return guardados

    def responder(self, consulta: str) -> dict:
        CONTEXTO["cliente"] = self.cliente
        recuerdos = self.lp.recuperar(consulta, k=4, cliente=self.cliente)          # IL2.2: recuperación semántica
        texto_lp = "\n".join(f"- {r['texto']}" for r in recuerdos) or "(sin datos previos del cliente)"
        res = llamar_con_espera(self.executor.invoke, {
            "input": consulta,
            "chat_history": self.cp.historial_mensajes(),                           # IL2.2: ventana de corto plazo
            "memoria_largo": texto_lp,
            "resumen": self.cp.resumen or "(sin resumen aun)",
        })
        salida = res["output"]
        herramientas = [accion.tool for accion, _ in res.get("intermediate_steps", [])]
        guardados = self._memorizar_declaraciones(consulta, herramientas)
        self.cp.agregar_turno(consulta, salida)
        return {"respuesta": salida, "herramientas_usadas": herramientas,
                "pasos": [(accion.tool, str(obs)[:400]) for accion, obs in res.get("intermediate_steps", [])],
                "recuerdos_usados": [r["texto"] for r in recuerdos], "memorizado_automatico": guardados}


def mostrar(res):
    print("\n💬 RESPUESTA:", res["respuesta"])
    print("🛠️ Herramientas usadas:", res["herramientas_usadas"])
    if res["recuerdos_usados"]:
        print("🧠 Recuerdos inyectados:", res["recuerdos_usados"])


def responder_con_pausa(agente, texto, pausa_extra=60):
    """Si Mistral sigue saturado, espera un minuto completo y hace un ultimo intento."""
    try:
        return agente.responder(texto)
    except RuntimeError as e:
        print(f"⚠️ {e}\n   Esperando {pausa_extra}s antes de un ultimo intento...")
        time.sleep(pausa_extra)
        return agente.responder(texto)


print(f"✅ Agente listo con {len(TOOLS_AGENTE)} herramientas.")

## ▶️ Demostraciones

**Demo 1 – Planificación y decisiones adaptativas (IL2.3).** Cuatro pedidos que obligan al planificador a cambiar su comportamiento. No usan LLM, por lo que el resultado es reproducible y la bitácora muestra cada decisión.

| Caso | Condición que cambia | Decisión esperada |
|---|---|---|
| A | Configuración AM5 completa y compatible | Emitir cotización sin replanificar |
| B | Procesador AM5 + placa AM4 + RAM DDR5 | Replanificar: reemplazar la **placa** por la B650 |
| C | RTX 4070 con fuente de 450W | Replanificar: reemplazar la **fuente** por la Corsair 650W |
| D | Configuración AM4 con RAM DDR4 **sin stock** | Sin alternativa compatible → **derivar** a vendedor |

In [ ]:
# --- Demo 1: planificación con replanificación reactiva (sin LLM, determinista) ---
CASOS_PLANIFICACION = {
    "A) Todo compatible": "Cotiza un PC con Ryzen 7 7800X3D, placa ASUS B650, RAM Kingston DDR5, RTX 4070 y fuente Corsair RM650x.",
    "B) Placa incompatible": "Quiero armar un PC con Ryzen 7 7800X3D, la placa MSI B450 y la RAM Kingston DDR5.",
    "C) Fuente insuficiente": "Arma una cotización con Ryzen 7 7800X3D, placa B650, RAM DDR5, RTX 4070 y la fuente EVGA de 450W.",
    "D) Sin stock y sin alternativa": "Necesito un PC con Ryzen 5 5600X, placa MSI B450 y memoria DDR4 Corsair.",
}

resultados_demo1 = {}
for nombre, pedido in CASOS_PLANIFICACION.items():
    print("\n" + "=" * 90 + f"\nCASO {nombre}\n" + "=" * 90)
    res = planificador.ejecutar(pedido)
    resultados_demo1[nombre] = res
    print("\n--- Tabla del plan final ---")
    mostrar_plan(res["plan"])
    print(f"\nEstado final: {res['estado']} | Plan v{res['version_plan']} | Sustituciones: {res['sustituciones']}")

**Demo 2 – Memoria de corto y largo plazo (IL2.2).** Conversación larga con ventana de solo 2 turnos (así se activa el resumen) y luego una **sesión nueva**: el agente no tiene el historial, pero recupera los datos del cliente desde la memoria de largo plazo.

In [ ]:
# --- Demo 2: memoria de corto plazo (ventana + resumen) y largo plazo (entre sesiones) ---
CLIENTE_DEMO = "cliente-camila"
memoria_lp.entradas = [e for e in memoria_lp.entradas if e["cliente"] != CLIENTE_DEMO]   # demo reproducible

print("##### SESIÓN 1 (ventana de 2 turnos) #####")
sesion1 = AgenteSPDigital(cliente=CLIENTE_DEMO, k_ventana=2)
mensajes_s1 = [
    "Hola, soy Camila. Tengo una placa MSI B450 Tomahawk Max y una fuente EVGA de 450W. Mi presupuesto es de 800000 CLP.",
    "¿Qué procesador de tu catálogo me sirve para mi placa?",
    "¿Y qué memoria RAM me recomiendas con ese procesador?",
    "¿Tiene stock esa memoria?",
]
for m in mensajes_s1:
    print(f"\n👤 Cliente: {m}")
    r = responder_con_pausa(sesion1, m)
    mostrar(r)
    time.sleep(15)

print("\n📝 Resumen acumulado (turnos que salieron de la ventana):\n", sesion1.cp.resumen)
print("🧠 Hechos guardados en largo plazo:", memoria_lp.hechos_de(CLIENTE_DEMO))

print("\n\n##### SESIÓN 2: conversación NUEVA (memoria de corto plazo vacía) #####")
sesion2 = AgenteSPDigital(cliente=CLIENTE_DEMO)
pregunta = "Quiero agregar la RTX 4070 a mi equipo. ¿Tendría que cambiar algo de lo que ya tengo?"
print(f"\n👤 Cliente: {pregunta}")
r2 = responder_con_pausa(sesion2, pregunta)
mostrar(r2)

**Demo 3 – Agente completo en un flujo organizacional simulado (IL2.1 + IL2.3).** Un mismo agente decide qué herramienta usar según el pedido: razonar con RAG, delegar en el planificador o derivar a un humano.

In [ ]:
# --- Demo 3: el agente elige herramientas según el pedido ---
print("⏳ Pausa de 30s para que se limpie la ventana de Mistral tras la Demo 2...")
time.sleep(30)
agente_demo = AgenteSPDigital(cliente="cliente-demo-3")
FLUJO_DEMO = [
    ("Compatibilidad simple",      "¿Es compatible el Ryzen 7 7800X3D con la placa MSI B450 Tomahawk Max?"),
    ("Cotización multi-paso",      "Cotízame un PC con Ryzen 7 7800X3D, la placa MSI B450 y la RAM Kingston DDR5."),
    ("Fuera del catálogo",         "¿Tienen disponibilidad de un cooler líquido de 360mm para gabinete ITX?"),
]
for titulo, pedido in FLUJO_DEMO:
    print("\n" + "=" * 90 + f"\n{titulo}\n" + "=" * 90 + f"\n👤 Cliente: {pedido}")
    mostrar(responder_con_pausa(agente_demo, pedido))
    time.sleep(15)

print("\n📁 Archivos escritos por el agente:")
for ruta in sorted(DIR_COTIZACIONES.glob("*.json"))[-3:]:
    print("  -", ruta)
print("  -", RUTA_DERIVACIONES, f"({sum(1 for _ in open(RUTA_DERIVACIONES, encoding='utf-8'))} tickets)" if RUTA_DERIVACIONES.exists() else "(sin tickets)")

## ✅ Pruebas de software y evaluación del agente

1. **Pruebas automáticas (sin API)**: verifican herramientas, memoria y planificador con `assert`. Usan una carpeta temporal y *embeddings* falsos, así que no gastan tokens ni tocan `datos_agente/`.
2. **Evaluación del agente (con LLM)**: mide si el agente **elige la herramienta correcta** y si la **decisión final** es la esperada; los resultados también se suben a LangSmith.

In [ ]:
# --- Pruebas automáticas (sin llamadas a la API): herramientas, memoria y planificador ---
import tempfile, zlib

def _embed_falso(textos):
    """Embedding determinista por bolsa de palabras (solo para pruebas)."""
    out = []
    for t in textos:
        v = np.zeros(64, dtype="float32")
        for w in re.findall(r"\w+", t.lower()):
            v[zlib.crc32(w.encode()) % 64] += 1
        out.append(v)
    return np.array(out)

def _ejecutor_local(nombre, args):
    return HERRAMIENTAS_FN[nombre](**args)

pruebas = []
def prueba(fn):
    pruebas.append(fn)
    return fn

# ---------- Herramientas ----------
@prueba
def test_reglas_son_simetricas():
    assert evaluar_par("MB-B450-MSI", "CPU-R7-7800X3D")["veredicto"] == "NO COMPATIBLE"
    assert evaluar_par("CPU-R7-7800X3D", "MB-B450-MSI")["veredicto"] == "NO COMPATIBLE"

@prueba
def test_reglas_originales_se_mantienen():
    for par, regla in REGLAS_COMPATIBILIDAD.items():
        assert REGLAS_AGENTE[par] == regla

@prueba
def test_par_sin_regla_deriva():
    assert evaluar_par("CPU-R5-5600X", "SKU-INEXISTENTE")["veredicto"] == "SIN REGLA -> DERIVAR"

@prueba
def test_extraer_skus_lenguaje_natural():
    skus = extraer_skus("Quiero un Ryzen 7 7800X3D con placa B650 y RAM DDR5")
    assert set(skus) == {"CPU-R7-7800X3D", "MB-B650-ASUS", "RAM-DDR5-32GB-KINGSTON"}

@prueba
def test_stock_y_producto_inexistente():
    assert json.loads(consultar_stock_precio("GPU-4070-ASUS"))["disponible"] is True
    assert json.loads(consultar_stock_precio("RAM-DDR4-16GB-CORSAIR"))["disponible"] is False
    assert json.loads(consultar_stock_precio("cooler liquido 360mm"))["existe"] is False

@prueba
def test_buscar_alternativa_placa():
    r = json.loads(buscar_alternativa("MB-B450-MSI", "CPU-R7-7800X3D,RAM-DDR5-32GB-KINGSTON"))
    assert [c["sku"] for c in r["candidatos"]] == ["MB-B650-ASUS"]

@prueba
def test_buscar_alternativa_inexistente_devuelve_lista_vacia():
    r = json.loads(buscar_alternativa("RAM-DDR4-16GB-CORSAIR", "MB-B450-MSI,CPU-R5-5600X"))
    assert r["candidatos"] == []          # la unica otra RAM es DDR5 (incompatible) -> no hay alternativa

@prueba
def test_cotizacion_rechaza_incompatibles_y_emite_validas():
    global DIR_COTIZACIONES, RUTA_DERIVACIONES
    original = DIR_COTIZACIONES
    with tempfile.TemporaryDirectory() as tmp:
        DIR_COTIZACIONES = Path(tmp)
        try:
            malo = json.loads(generar_cotizacion("CPU-R7-7800X3D,MB-B450-MSI"))
            assert malo["estado"] == "RECHAZADA" and not list(Path(tmp).glob("*.json"))
            bueno = json.loads(generar_cotizacion("CPU-R7-7800X3D,MB-B650-ASUS,RAM-DDR5-32GB-KINGSTON"))
            assert bueno["estado"] == "EMITIDA" and bueno["total_clp"] == 379990 + 189990 + 119990
            assert len(list(Path(tmp).glob("COT-*.json"))) == 1
        finally:
            DIR_COTIZACIONES = original

@prueba
def test_derivacion_escribe_ticket():
    global RUTA_DERIVACIONES
    original = RUTA_DERIVACIONES
    with tempfile.TemporaryDirectory() as tmp:
        RUTA_DERIVACIONES = Path(tmp) / "d.jsonl"
        try:
            r = json.loads(derivar_a_vendedor("sin regla", "cooler 360mm"))
            lineas = RUTA_DERIVACIONES.read_text(encoding="utf-8").strip().splitlines()
            assert r["estado"] == "DERIVADO" and json.loads(lineas[0])["consulta"] == "cooler 360mm"
        finally:
            RUTA_DERIVACIONES = original

# ---------- Memoria ----------
@prueba
def test_memoria_corto_plazo_ventana_y_resumen():
    cp = MemoriaCortoPlazo(resumidor=lambda res, u, a: (res + " | " + u).strip(" |"), k=2)
    for i in range(4):
        cp.agregar_turno(f"pregunta {i}", f"respuesta {i}")
    assert len(cp.turnos) == 2 and cp.turnos[0][0] == "pregunta 2"
    assert "pregunta 0" in cp.resumen and "pregunta 1" in cp.resumen
    assert len(cp.historial_mensajes()) == 4

@prueba
def test_memoria_corto_plazo_respaldo_si_falla_el_llm():
    def roto(*a): raise RuntimeError("429")
    cp = MemoriaCortoPlazo(resumidor=roto, k=1)
    cp.agregar_turno("hola", "buenas"); cp.agregar_turno("segunda", "ok")
    assert "hola" in cp.resumen

@prueba
def test_memoria_largo_plazo_persistencia_aislamiento_y_semantica():
    with tempfile.TemporaryDirectory() as tmp:
        ruta = Path(tmp) / "mem.json"
        m = MemoriaLargoPlazo(ruta, embed_fn=_embed_falso)
        m.agregar("Tengo una placa MSI B450 Tomahawk", cliente="c1")
        m.agregar("Mi presupuesto es 800000 CLP", cliente="c1")
        m.agregar("Tengo una fuente EVGA", cliente="c2")
        assert m.agregar("Tengo una placa MSI B450 Tomahawk", cliente="c1") == m.entradas[0]["id"]   # sin duplicados
        assert len(m.entradas) == 3
        top = m.recuperar("que placa MSI tiene", k=1, cliente="c1")
        assert "B450" in top[0]["texto"]
        assert all(r["cliente"] == "c1" for r in m.recuperar("fuente", k=5, cliente="c1"))        # aislamiento
        nueva_sesion = MemoriaLargoPlazo(ruta, embed_fn=_embed_falso)                                # persistencia
        assert len(nueva_sesion.entradas) == 3 and nueva_sesion.hechos_de("c2") == ["Tengo una fuente EVGA"]

# ---------- Planificador ----------
def _planificar(pedido):
    global DIR_COTIZACIONES, RUTA_DERIVACIONES
    o1, o2 = DIR_COTIZACIONES, RUTA_DERIVACIONES
    with tempfile.TemporaryDirectory() as tmp:
        DIR_COTIZACIONES, RUTA_DERIVACIONES = Path(tmp), Path(tmp) / "d.jsonl"
        try:
            return PlanificadorCotizacion(ejecutor=_ejecutor_local, verbose=False).ejecutar(pedido)
        finally:
            DIR_COTIZACIONES, RUTA_DERIVACIONES = o1, o2

@prueba
def test_plan_todo_compatible_no_replanifica():
    r = _planificar(CASOS_PLANIFICACION["A) Todo compatible"])
    assert r["estado"] == "COTIZACION_EMITIDA" and r["sustituciones"] == [] and r["version_plan"] == 1

@prueba
def test_plan_reemplaza_placa_incompatible():
    r = _planificar(CASOS_PLANIFICACION["B) Placa incompatible"])
    assert r["estado"] == "COTIZACION_EMITIDA" and r["version_plan"] == 2
    assert r["sustituciones"][0]["reemplaza"] == "MB-B450-MSI" and r["sustituciones"][0]["por"] == "MB-B650-ASUS"
    assert "MB-B450-MSI" not in r["skus_finales"]

@prueba
def test_plan_reemplaza_fuente_insuficiente():
    r = _planificar(CASOS_PLANIFICACION["C) Fuente insuficiente"])
    assert r["estado"] == "COTIZACION_EMITIDA"
    assert r["sustituciones"][0] == {"reemplaza": "PSU-EVGA-450", "por": "PSU-CORSAIR-650", "ronda": 1}

@prueba
def test_plan_sin_stock_sin_alternativa_deriva():
    r = _planificar(CASOS_PLANIFICACION["D) Sin stock y sin alternativa"])
    assert r["estado"] == "DERIVADO" and r["resultado_final"]["estado"] == "DERIVADO"

@prueba
def test_plan_producto_fuera_de_catalogo_deriva():
    assert _planificar("Quiero un cooler liquido de 360mm para gabinete ITX")["estado"] == "DERIVADO"

@prueba
def test_plan_respeta_prioridades_y_dependencias():
    r = _planificar(CASOS_PLANIFICACION["A) Todo compatible"])
    pasos = r["plan"].pasos
    assert pasos[-1].accion == "generar_cotizacion" and pasos[-1].prioridad == 3          # lo de baja prioridad va al final
    for p in pasos:
        for d in p.dependencias:
            assert d < p.id                                                                # dependencias siempre previas

fallos = 0
for t in pruebas:
    try:
        t()
        print(f"  ✅ {t.__name__}")
    except Exception as e:
        fallos += 1
        print(f"  ❌ {t.__name__}: {type(e).__name__}: {e}")
print(f"\nResultado: {len(pruebas) - fallos}/{len(pruebas)} pruebas OK")
assert fallos == 0, "Hay pruebas fallidas"

In [ ]:
# --- Evaluación del agente con LLM: ¿elige la herramienta correcta y toma la decisión esperada? ---
CASOS_AGENTE = [
    {"id": "compatibilidad_simple",
     "pregunta": "¿Es compatible el Ryzen 5 5600X con la placa MSI B450 Tomahawk Max?",
     "herramientas_aceptadas": ["verificar_compatibilidad", "razonar_con_rag"], "estado_esperado": None},
    {"id": "consulta_stock",
     "pregunta": "¿Cuánto cuesta y hay stock de la RTX 4070?",
     "herramientas_aceptadas": ["consultar_stock_precio"], "estado_esperado": None},
    {"id": "cotizacion_con_reemplazo",
     "pregunta": "Cotízame un PC con Ryzen 7 7800X3D, la placa MSI B450 y la RAM Kingston DDR5.",
     "herramientas_aceptadas": ["ejecutar_plan_cotizacion"], "estado_esperado": "COTIZACION_EMITIDA"},
    {"id": "cotizacion_sin_alternativa",
     "pregunta": "Necesito cotizar un PC con Ryzen 5 5600X, placa MSI B450 y memoria DDR4 Corsair.",
     "herramientas_aceptadas": ["ejecutar_plan_cotizacion"], "estado_esperado": "DERIVADO"},
    {"id": "fuera_de_catalogo",
     "pregunta": "¿Tienen disponibilidad de un cooler líquido de 360mm para gabinete ITX?",
     "herramientas_aceptadas": ["derivar_a_vendedor"], "estado_esperado": None},
]

aciertos_herramienta, aciertos_decision, con_estado = 0, 0, 0
for caso in CASOS_AGENTE:
    try:
        ag = AgenteSPDigital(cliente="cliente-evaluacion", verbose=False)
        res = ag.responder(caso["pregunta"])
        usadas = res["herramientas_usadas"]
        ok_h = any(h in usadas for h in caso["herramientas_aceptadas"])
        ok_d = None
        if caso["estado_esperado"]:
            con_estado += 1
            obs = " ".join(o for t, o in res["pasos"] if t == "ejecutar_plan_cotizacion")
            ok_d = caso["estado_esperado"] in obs
            aciertos_decision += int(ok_d)
        aciertos_herramienta += int(ok_h)
        print(f"[{'OK' if ok_h else 'FALLA'}] {caso['id']:<28} herramientas={usadas}"
              + (f" | decisión {'OK' if ok_d else 'FALLA'} (esperada: {caso['estado_esperado']})" if ok_d is not None else ""))
    except Exception as e:
        print(f"[ERROR] {caso['id']}: {type(e).__name__}: {e}")
    time.sleep(15)

print(f"\nSelección de herramienta correcta: {aciertos_herramienta}/{len(CASOS_AGENTE)} ({100*aciertos_herramienta/len(CASOS_AGENTE):.0f}%)")
if con_estado:
    print(f"Decisión final correcta (planificación): {aciertos_decision}/{con_estado} ({100*aciertos_decision/con_estado:.0f}%)")
print("Las ejecuciones del agente quedan trazadas en LangSmith (proyecto SP-Digital-RAG-Evaluation).")